# チュートリアル 05: マルチエージェントワークフロー (基礎)

##  学習目標
このノートブックを完了すると、以下ができるようになります:
- マルチエージェント調整のための Agent Framework の **組み込みワークフロービルダー** を使用する
- `SequentialBuilder` で **順次ワークフロー** を構築する (エージェントチェーン)
- `ConcurrentBuilder` で **並行ワークフロー** を構築する (並列実行)
- 各基本ワークフローパターンをいつ使用するかを理解する
- 高度なパターンの基礎を学ぶ (チュートリアル 07 で説明)

##  主要概念

### なぜマルチエージェントシステム?

**単一エージェント (チュートリアル 1-5):**
-  シンプル、1 つのエージェントがすべてを処理
-  あまりにも多くの責任を持つと過負荷になる可能性
-  特化や並列化がない
- 例: 汎用アシスタント

**マルチエージェントワークフロー:**
-  複数の専門エージェントが協力して作業
-  各エージェントは自分の専門分野に集中
-  速度のための並列実行
-  明確な関心の分離
- 例: フライト専門家 + ホテル専門家 + アクティビティ専門家

### Agent Framework ワークフローパターン

Agent Framework は **組み込みワークフロービルダー** を提供します:

**基本パターン (このチュートリアル):**

1. **`SequentialBuilder`** - エージェントが順番に作業 (A → B → C)
   - 共有される会話コンテキストが各エージェントを通過
   - 各エージェントは前のエージェントの出力に基づいて構築
   - 用途: レビュー/改良、多段階タスク

2. **`ConcurrentBuilder`** - エージェントが並列に作業 (A, B, C 同時に)
   - すべてのエージェントに同じ入力でファンアウト
   - すべてのエージェントからの結果を集約するファンイン
   - 用途: 独立したタスク、複数の視点の収集

**高度なパターン (チュートリアル 07):**

3. **`WorkflowBuilder`** - 実行者とエッジを持つカスタム DAG
   - ワークフローグラフの完全な制御
   - カスタムルーティングロジックと条件
   - 用途: 複雑なビジネスロジック、条件付きフロー

4. **`MagenticBuilder`** - AI を利用したオーケストレーション
   - LLM が実行プランを作成して管理
   - 動的なエージェント選択と調整
   - 用途: 複雑で予測不可能なタスク

### Framework ワークフローの利点

 **組み込み調整** - 手動のオーケストレーションコード不要  
 **自動集約** - 結果が自動的に結合される  
 **イベントストリーミング** - リアルタイムで進捗を追跡  
 **チェックポイント** - 長いワークフローの一時停止/再開  
 **可視化** - ワークフローグラフを表示  
 **本番環境対応** - エラーハンドリング、可観測性が組み込まれている

---

## ステップ 1: セットアップとインポート

In [ ]:
import asyncio
from typing import cast

from agent_framework import (
    # 進捗追跡のためのワークフローイベント
    WorkflowEvent,
    WorkflowEventType,
    # 基本型
    Message,
    WorkflowViz,
)
from agent_framework.orchestrations import (
    # 基本ワークフロービルダー
    SequentialBuilder,
    ConcurrentBuilder,
)
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

# 環境変数の読み込み
load_dotenv(override=True)
print("✅ インポート成功!")
print("📦 ワークフロービルダー準備完了: SequentialBuilder、ConcurrentBuilder")
print("💡 高度なパターン (WorkflowBuilder、MagenticBuilder) については、チュートリアル 07 を参照してください!")

In [ ]:
import os
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === 認証方式の選択 ===
# True: APIキー認証, False: DefaultAzureCredential 認証
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("🔑 認証方式: APIキー認証")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # フレームワークが環境変数の AZURE_OPENAI_API_KEY を自動読み込みして
    # credential より優先してしまうため、明示的に削除する
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 認証方式: DefaultAzureCredential")

print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

## OpenTelemetry によるトレーサーのセット
マルチエージェントのデバッグには OpenTelemetry によるトレーサーを利用すると便利です。
この環境の Agent Framework では `setup_observability` は提供されていないため、
OpenTelemetry の `TracerProvider`（Exporter/Processor 含む）を手動でセットし、`enable_instrumentation()` で計測を有効化します。
ここではトレースの送信先として OTLP gRPC（例: Jaeger / OpenTelemetry Collector の `4317`）を使います。

Jaeger UI: [http://localhost:16686](http://localhost:16686)

In [ ]:
import os

from agent_framework.observability import configure_otel_providers, get_tracer

service_name = "agent_framework"
otlp_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4317")

# 環境変数ベースで設定（Agent Framework は標準の OTEL_* を読む）
os.environ.setdefault("OTEL_SERVICE_NAME", service_name)
os.environ.setdefault("OTEL_EXPORTER_OTLP_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_PROTOCOL", "grpc")
os.environ.setdefault("ENABLE_INSTRUMENTATION", "true")
os.environ.setdefault("OTEL_METRICS_EXPORTER", "none")  # Metrics は無効化（必要に応じて変更）

# enable_sensitive_data=True を指定して機微データ(OpenAI の IN/OUT データ)の収集を有効化
configure_otel_providers(enable_sensitive_data=True)

tracer = get_tracer()
print(f"Observability configured: service={service_name} otlp={otlp_endpoint}")

## ステップ 2: 専門エージェントを作成

ワークフローを使用して調整するドメイン専門家エージェントを作成しましょう。

In [ ]:
async def create_travel_agents():
    """
    専門旅行計画エージェントを作成します。
    """
    # Azure OpenAI クライアントの作成
    chat_client = AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment,
    )

    # フライト専門家
    flight_agent = chat_client.as_agent(
        instructions="""
        あなたは専門のフライト予約スペシャリストです。
        以下を考慮した簡潔で実用的なフライト推奨を提供してください:
        - 予約に最適な時期
        - 航空会社の好みと品質
        - 乗り継ぎ戦略
        - 価格と利便性のトレードオフ
        
        回答は簡潔に (最大2-3文) してください。
        """,
        name="FlightExpert",
    )
    
    # ホテル専門家
    hotel_agent = chat_client.as_agent(
        instructions="""
        あなたは専門のホテル予約スペシャリストです。
        以下を考慮した簡潔なホテル推奨を提供してください:
        - 観光客に最適なエリア
        - コストパフォーマンス
        - アトラクションや交通機関への近さ
        - ホテルの質とアメニティ
        
        回答は簡潔に (最大2-3文) してください。
        """,
        name="HotelExpert",
    )
    
    # アクティビティ専門家
    activities_agent = chat_client.as_agent(
        instructions="""
        あなたは地元のアクティビティと体験の専門家です。
        以下を考慮した簡潔なアクティビティ推奨を提供してください:
        - 必見のアトラクション
        - 地元の人気スポットと隠れた名所
        - 文化的体験
        - 食事とグルメ
        
        回答は簡潔に (最大2-3文) してください。
        """,
        name="ActivitiesExpert",
    )
    
    return flight_agent, hotel_agent, activities_agent

print("✅ エージェントファクトリー作成完了")

In [ ]:
from __future__ import annotations
import sys
from contextlib import nullcontext
from typing import Optional, cast
from agent_framework import AgentResponse, AgentResponseUpdate, Message, WorkflowEvent

async def run_stream_pretty(
    workflow,
    task: str,
    *,
    tracer=None,
    span_name: str = "SequentialBuilder",
    print_final: bool = True,
) -> list[Message]:
    """
    
    workflow.run(task, stream=True) を実行し、ストリーミング出力を
    Jupyter Notebook 上でリアルタイムに表示するヘルパー。

    前提（GroupChatBuilder 使用時）:
      GroupChatBuilder(..., intermediate_outputs=True) でビルドすること。
      デフォルト(False)ではオーケストレーターの output のみが yield され、
      参加者の AgentResponseUpdate はフィルタされる。

    表示:
      - AgentResponseUpdate → トークンを逐次表示（sys.stdout.write + flush）
      - executor 切替時 → 改行 + 名前ヘッダ
      - 最終 list[Message] → CONVERSATION LOG としてまとめ表示

    注意:
      ★ start_as_current_span() を使用し、ワークフロー内部のスパンを
      このスパンの子として正しくネストさせる。
      _process_events は通常の async 関数であり async generator ではないため、
      コンテキストマネージャと組み合わせても GeneratorExit 問題は発生しない。
    """
    final_conversation: list[Message] = []
    last_executor_id: Optional[str] = None

    def _write(text: str) -> None:
        """Jupyter でも確実にフラッシュする書き込み"""
        sys.stdout.write(text)
        sys.stdout.flush()

    async def _process_events(workflow, task):
        nonlocal final_conversation, last_executor_id
        async for event in workflow.run(task, stream=True):
            if not isinstance(event, WorkflowEvent):
                continue

            # ----- executor 切替通知 -----
            if event.type == "executor_invoked":
                eid = event.executor_id
                if eid != last_executor_id:
                    if last_executor_id is not None:
                        _write("\n")
                    _write(f"🤖 {eid}: ")
                    last_executor_id = eid

            # ----- output イベント -----
            elif event.type == "output":
                data = event.data

                # (1) ストリーミングチャンク: AgentResponseUpdate
                if isinstance(data, AgentResponseUpdate):
                    eid = event.executor_id
                    if eid != last_executor_id:
                        if last_executor_id is not None:
                            _write("\n")
                        _write(f"🤖 {eid}: ")
                        last_executor_id = eid
                    chunk = data.text or ""
                    if chunk:
                        _write(chunk)

                # (2) 非ストリーミング応答: AgentResponse
                elif isinstance(data, AgentResponse):
                    eid = event.executor_id
                    if eid != last_executor_id:
                        if last_executor_id is not None:
                            _write("\n")
                        _write(f"🤖 {eid}: ")
                        last_executor_id = eid
                    text = data.text or ""
                    if text:
                        _write(text)

                # (3) 最終会話ログ: list[Message]
                elif isinstance(data, list):
                    final_conversation = cast(list[Message], data)

    # ★ start_as_current_span() を使用してワークフロー内部スパンを子としてネスト。
    #   _process_events は通常の async 関数なのでコンテキスト競合は発生しない。
    cm = tracer.start_as_current_span(span_name) if tracer else nullcontext()
    with cm:
        await _process_events(workflow, task)

    # ストリーム末尾の改行
    _write("\n")

    if print_final and final_conversation:
        print("\n" + "=" * 80)
        print("CONVERSATION LOG")
        print("=" * 80)
        for msg in final_conversation:
            author = getattr(msg, "author_name", None) or msg.role
            text = msg.text or str(msg)
            print(f"\n[{author}]")
            print(text)
            print("-" * 80)

    return final_conversation

In [ ]:
from agent_framework import WorkflowViz
from IPython.display import SVG, display
import os

def visualize_workflow(workflow, filename="workflow_diagram", show_preview=True):
    """
    ワークフローのグラフ情報を出力し、SVGで保存し、プレビューする関数
    """
    viz = WorkflowViz(workflow)
    
    try:
        svg_path = viz.export(format="svg", filename=filename)
        print("=" * 60)
        print(f"✅ SVGファイルを保存しました: {svg_path}")
        print("=" * 60)
        print()
        
        if show_preview and os.path.exists(svg_path):
            display(SVG(filename=svg_path))
        
        return svg_path
        
    except ImportError as e:
        print("❌ エラー: 'graphviz'パッケージがインストールされていません")
        print("インストール方法: pip install agent-framework[viz] --pre")
        print(f"詳細: {e}")
        return None
    except Exception as e:
        print(f"❌ エラーが発生しました: {e}")
        return None

## ステップ 3: 順次ワークフロー - SequentialBuilder

**順次パターン** は、各エージェントが前のエージェントの出力に基づいて構築するようにエージェントをチェーンします。

**ユースケース:**
- 執筆 → レビュー → 編集
- 研究 → 分析 → 推奨事項
- ドラフト → 批評 → 改良

共有される会話は、各参加者を順次通過します。

In [ ]:
async def sequential_workflow_demo():
    """
    SequentialBuilder を実演: エージェントが順番に作業。
    各エージェントは前のエージェントからの完全な会話履歴を表示します。
    """
    print("=== 順次ワークフローパターン ===")
    print("エージェントが順番に作業: フライト → ホテル → アクティビティ")
    print("各エージェントは前のエージェントの推奨事項に基づいて構築\n")
    
    flight_agent, hotel_agent, activities_agent = await create_travel_agents()
    
    # SequentialBuilder を使用して順次ワークフローを構築
    workflow = SequentialBuilder(
        participants=[flight_agent, hotel_agent, activities_agent]
    ).build()
    
    print("🚀 順次ワークフロー実行中...\n")
    
    # ワークフローを実行して出力を収集
    outputs: list[list[Message]] = []
    async for event in workflow.run(
        "週末のパリ旅行を計画してください。フライト、ホテル、文化的なアクティビティを希望します。",
        stream=True,
    ):
        # エージェントの進捗を追跡
        if event.type == "executor_invoked":
            print(f"   🤖 {event.executor_id} が作業中...")
        
        # 最終出力を収集
        if event.type == "output":
            outputs.append(cast(list[Message], event.data))
    
    # 最終会話を表示
    if outputs:
        print(f"\n{'='*60}")
        print("📋 最終会話 (順次フロー)")
        print(f"{'='*60}\n")
        
        for i, msg in enumerate(outputs[-1], start=1):
            name = msg.author_name or msg.role
            print(f"{'-'*60}")
            print(f"{i}. [{name}]")
            print(f"{'-'*60}")
            print(f"{msg.text}\n")
        
        print(f"{'='*60}")
        print("✅ 順次ワークフロー完了!")
        print(f"{'='*60}")
        print("利点:")
        print("✓ 各エージェントが前の推奨事項に基づいて構築")
        print("✓ 共有される会話コンテキスト")
        print("✓ 手動調整コード不要")
        print("✓ 明確な順次フロー")

await sequential_workflow_demo()

In [ ]:

flight_agent, hotel_agent, activities_agent = await create_travel_agents()

# SequentialBuilder を使用して順次ワークフローを構築
# ★ intermediate_outputs=True: 各エージェントの AgentResponseUpdate（トークンチャンク）を
#   output イベントとして yield する。False(デフォルト)では "end" エグゼキュータの
#   list[Message] のみが通過し、ストリーミング出力が一切表示されない。
workflow = SequentialBuilder(
    participants=[flight_agent, hotel_agent, activities_agent],
    intermediate_outputs=True,
).build()

# 関数を使ってワークフローを可視化
svg_path = visualize_workflow(
    workflow, 
    filename="sequential_multi_agent_workflow.svg",
    show_preview=True
)


In [ ]:
task = "週末のパリ旅行を計画してください。フライト、ホテル、文化的なアクティビティを希望します。"

# run_stream をヘルパーで実行（stream表示 + 最終会話の回収）
final_conversation = await run_stream_pretty(workflow, task, tracer=tracer)

In [ ]:
for msg in final_conversation:
    print(msg.to_dict())

## ステップ 4: 並行ワークフロー - ConcurrentBuilder

**並行パターン** は、自動ファンアウト/ファンインでエージェントを並列実行します。

**ユースケース:**
- 複数の視点を同時に収集
- 独立した並列タスク
- 研究者 + マーケティング担当者 + 法務レビュー (異なるドメイン)

すべてのエージェントが同じ入力を受け取り、同時に作業します。

In [ ]:

"""
ConcurrentBuilder を実演: エージェントが並列に作業。
すべてのエージェントが同じプロンプトを同時に受信。
結果は自動的に集約されます。
"""
print("=== 並行ワークフローパターン ===")
print("エージェントが並列に作業: フライト ∥ ホテル ∥ アクティビティ")
print("すべてのエージェントが同じ入力を受信、同時に作業\n")

flight_agent, hotel_agent, activities_agent = await create_travel_agents()

# ConcurrentBuilder を使用して並行ワークフローを構築
# ★ intermediate_outputs=True: 各エージェントの AgentResponseUpdate（トークンチャンク）を
#   output イベントとして yield する。False(デフォルト)では "aggregator" の
#   list[Message] のみが通過し、ストリーミング出力が一切表示されない。
workflow = ConcurrentBuilder(
    participants=[flight_agent, hotel_agent, activities_agent],
    intermediate_outputs=True,
).build()


# 関数を使ってワークフローを可視化
svg_path = visualize_workflow(
    workflow, 
    filename="concurrent_multi_agent_workflow.svg",
    show_preview=True
)


In [ ]:

print("🚀 並行ワークフロー実行中 (すべてのエージェントが並列)...\n")

# ★ start_as_current_span() を使用してワークフロー内部スパンを子としてネスト。
with tracer.start_as_current_span("ConcurrentBuilder"):
    # ワークフローを実行して出力を収集
    outputs: list[list[Message]] = []
    async for event in workflow.run(
        "3日間の東京旅行を計画しています。フライト、ホテル、やるべきことの推奨を提供してください。",
        stream=True,
    ):
        # エージェントの進捗を追跡
        if event.type == "executor_invoked":
            print(f"   🤖 {event.executor_id} が作業中...")
        
        # 最終出力を収集
        if event.type == "output":
            outputs.append(cast(list[Message], event.data))

    # 最終会話を表示
    if outputs:
        print(f"\n{'='*60}")
        print("📋 最終会話 (並行フロー)")
        print(f"{'='*60}\n")
        
        for i, msg in enumerate(outputs[-1], start=1):
            name = msg.author_name or msg.role
            print(f"{'-'*60}")
            print(f"{i}. [{name}]")
            print(f"{'-'*60}")
            print(f"{msg.text}\n")

##  ワークフローパターンの比較

### 各パターンをいつ使用するか

| パターン | ビルダー | ユースケース | 長所 | 短所 |
|---------|---------|----------|------|------|
| **順次** | `SequentialBuilder` | レビュー/改良、多段階タスク | シンプル、明確なフロー | 遅い、並列処理なし |
| **並行** | `ConcurrentBuilder` | 独立したタスク、複数の視点 | 速い、並列 | 順次依存関係なし |

### 実世界の例

**順次:**
- ドキュメント: ドラフト → レビュー → 編集 → 承認
- 研究: 収集 → 分析 → 要約 → 推奨
- コード: 作成 → テスト → レビュー → デプロイ

**並行:**
- 製品発売: マーケティング + 法務 + エンジニアリング (並列レビュー)
- 研究: 複数の研究者が異なる側面を調査
- 分析: 技術 + ビジネス + 法務の視点

### 高度なパターン (チュートリアル 07)

より複雑なシナリオについては、**チュートリアル 07: 高度なワークフロー** を参照してください:
- **WorkflowBuilder** - カスタム DAG、条件付きルーティング、検証ゲート
- **MagenticBuilder** - AI を利用したオーケストレーション、動的計画

##  重要なポイント

### 学んだこと

1. **組み込みワークフロービルダーを使用**
   - 手動調整コードを書かない!
   - 順次タスクには `SequentialBuilder`
   - 並列実行には `ConcurrentBuilder`

2. **順次ワークフロー**
   - エージェントが順番に作業: A → B → C
   - 各エージェントが完全な会話履歴を表示
   - 改良、多段階プロセスに最適
   - シンプルな `SequentialBuilder(participants=[agents]).build()` API

3. **並行ワークフロー**
   - エージェントが並列に作業: A ∥ B ∥ C
   - 自動ファンアウトと集約
   - 独立したタスク、速度に最適
   - 結果が自動的に結合される

4. **ワークフローが複雑さを処理**
   - 自動調整
   - 組み込みイベントストリーミング
   - エラーハンドリングと可観測性
   - 本番環境対応パターン

### 本番パターン

```python
from agent_framework.orchestrations import SequentialBuilder, ConcurrentBuilder

# 順次: レビューワークフロー
workflow = SequentialBuilder(
    participants=[writer, editor, approver]
).build()

# 並行: 並列分析
workflow = ConcurrentBuilder(
    participants=[technical_analyst, business_analyst, legal_analyst]
).build()

# ワークフロー実行 (stream=True で WorkflowEvent をストリーミング)
async for event in workflow.run("your prompt", stream=True):
    if event.type == "output":
        print(event.data)
```

##  演習問題

1. **コンテンツ作成パイプライン**
   - 順次: アイデアジェネレーター → ライター → エディター → SEO オプティマイザー
   - 各エージェントがコンテンツを改良

2. **製品分析**
   - 並行: 技術レビュアー ∥ 市場アナリスト ∥ 競合調査員
   - 複数の視点を同時に取得

3. **コードレビュー**
   - 順次: リンター → セキュリティチェッカー → パフォーマンスアナライザー → 承認者
   - 各ステップが前のチェックに基づいて構築

4. **顧客フィードバック分析**
   - 並行: 感情アナリスト ∥ 機能抽出 ∥ 優先度スコアラー
   - 異なる側面を並列に分析

In [ ]:
# 演習: コンテンツ作成ワークフローを作成

async def content_creation_exercise():
    """
    コンテンツ作成用の順次ワークフローを作成:
    アイデアジェネレーター → ライター → エディター → SEO オプティマイザー
    """
    # ここに実装を!
    # 1. 4つの専門エージェントを作成
    # 2. SequentialBuilder(participants=[agents]).build() で順次ワークフローを構築
    # 3. トピックで実行: workflow.run("topic", stream=True)
    # 4. 最終的に改良されたコンテンツを表示
    pass

async def product_analysis_exercise():
    """
    製品分析用の並行ワークフローを作成:
    技術 ∥ 市場 ∥ 競合 (並列)
    """
    # ここに実装を!
    # 1. 3つの専門アナリストエージェントを作成
    # 2. ConcurrentBuilder(participants=[agents]).build() で並行ワークフローを構築
    # 3. 製品説明で実行: workflow.run("product", stream=True)
    # 4. 集約された分析を表示
    pass

print(" 演習準備完了 - 順次および並行ワークフローを実装してください!")

##  次のステップ

おめでとうございます! 基本的なマルチエージェントワークフローパターンをマスターしました!

これで以下ができるようになりました:
-  順次エージェントワークフローを構築 (チェーン)
-  並行並列実行を作成
-  組み込みワークフロービルダーを使用
-  イベントでワークフローの進捗を追跡

---

### クイックリファレンス

**順次ワークフロー:**
```python
from agent_framework.orchestrations import SequentialBuilder

workflow = SequentialBuilder(
    participants=[agent1, agent2, agent3]
).build()

# ストリーミング実行
async for event in workflow.run("your prompt", stream=True):
    if event.type == "output":
        print(event.data)
```

**並行ワークフロー:**
```python
from agent_framework.orchestrations import ConcurrentBuilder

workflow = ConcurrentBuilder(
    participants=[agent1, agent2, agent3]
).build()

# 実行して出力を取得
result = await workflow.run("your prompt")
outputs = result.get_outputs()
```

**パターン選択:**
- **順次** → 順序が重要な場合、エージェントが互いに基づいて構築
- **並行** → 速度が重要な場合、独立したタスク
- **カスタム** → WorkflowBuilder についてはチュートリアル 07 を参照
- **Magentic** → AI オーケストレーションについてはチュートリアル 07 を参照